# Mesurer une distance focale

**Physique-chimie · PCSI / PSI**

Une lentille forme l’image d’un objet. On mesure les positions algébriques $p=\overline{OA}$ et $p'=\overline{OA'}$, en centimètres. La relation de conjugaison donne

$$\frac{1}{f'}=\frac{1}{p'}-\frac{1}{p} \qquad \Longrightarrow \qquad f'=\frac{pp'}{p-p'}.$$

**Objectif** — Estimer l’incertitude-type sur $f'$ par simulation Monte-Carlo et comparer à la propagation linéaire.

Exécute les trois cellules Python avec **Maj + Entrée**, ou lance **Exécuter toutes les cellules**. À la première ouverture, le démarrage de Python et le chargement des bibliothèques peuvent prendre quelques secondes.

## Simuler les mesures

Pour chaque essai, on tire indépendamment une valeur de $p$ et une valeur de $p'$ selon deux lois normales centrées sur les mesures. Les écarts-types sont $u(p)$ et $u(p')$. On calcule $f'$ pour chaque paire, puis la moyenne et l’écart-type des focales simulées.

Lis le code, puis repère l’expression qui calcule une focale pour chaque paire de mesures.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    # Au premier lancement dans JupyterLite, installer les commandes interactives.
    import piplite
    await piplite.install("ipywidgets==8.1.8")
    import ipywidgets as widgets
from IPython.display import display

def simuler_focales(p, p_image, u_p, u_image, n, graine=42):
    """Simule n couples de positions indépendantes et calcule les focales."""
    rng = np.random.default_rng(graine)
    positions_objet = rng.normal(p, u_p, n)
    positions_image = rng.normal(p_image, u_image, n)
    return positions_objet * positions_image / (positions_objet - positions_image)

## Comparer les deux expériences

Choisis un scénario dans le menu, puis clique sur **Simuler**. Les deux premiers scénarios reprennent les mesures de l’activité d’origine. Le troisième sert à étudier l’effet de la précision et peut aussi servir à saisir tes propres mesures.

La propagation linéaire, pour des mesures indépendantes, utilise les dérivées de $f'$ par rapport aux deux positions. Les valeurs sont calculées dans la cellule ci-dessous.

In [ ]:
def afficher_resultats(p, p_image, u_p, u_image, n):
    if p == p_image:
        print("Les deux positions coïncident. La formule ne donne pas de focale nominale.")
        return
    if u_p < 0 or u_image < 0:
        print("Les incertitudes-types doivent être positives ou nulles.")
        return

    ecart = p - p_image
    focale_nominale = p * p_image / ecart
    u_lineaire = np.hypot(p_image**2 * u_p, p**2 * u_image) / ecart**2

    focales = simuler_focales(p, p_image, u_p, u_image, n)
    if not np.isfinite(focales).all():
        print("Une paire de positions a donné une focale non finie.")
        return

    print(f"Focale nominale           {focale_nominale:.3g} cm")
    print(f"Moyenne Monte-Carlo       {np.mean(focales):.3g} cm")
    print(f"Incertitude Monte-Carlo   {np.std(focales, ddof=1):.3g} cm")
    print(f"Propagation linéaire      {u_lineaire:.3g} cm")
    if abs(ecart) <= 5 * np.hypot(u_p, u_image):
        print("Certaines paires peuvent approcher p = p′. Examine la sensibilité au tirage.")

    fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
    ax.hist(focales, bins=35, color="#147889", edgecolor="white")
    ax.axvline(focale_nominale, color="#9f3b24", linestyle="--", label="Focale nominale")
    ax.set(xlabel="Distance focale (cm)", ylabel="Nombre de tirages")
    ax.legend()
    plt.show()
    plt.close(fig)

In [ ]:
scenarios = {
    "Objet virtuel · activité d’origine": (25, 20, 0.5, 0.5),
    "Objet réel · activité d’origine": (-20, 20, 0.5, 0.5),
    "Objet réel · mesures moins précises": (-20, 20, 1.5, 1.5),
}
scenario = widgets.Dropdown(options=list(scenarios), description="Scénario",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="min(100%, 420px)"))
positions = [widgets.FloatText(description=nom, layout=widgets.Layout(width="155px"))
             for nom in ("p (cm)", "p′ (cm)", "u(p) (cm)", "u(p′) (cm)")]
tirages = widgets.BoundedIntText(value=1000, min=10, max=100000,
                                 description="Tirages", layout=widgets.Layout(width="155px"))
lancer = widgets.Button(description="Simuler", button_style="primary")
sortie = widgets.Output()

def choisir_scenario(_=None):
    for champ, valeur in zip(positions, scenarios[scenario.value]):
        champ.value = valeur

def lancer_simulation(_=None):
    with sortie:
        sortie.clear_output(wait=True)
        afficher_resultats(*(champ.value for champ in positions), tirages.value)

scenario.observe(choisir_scenario, names="value")
lancer.on_click(lancer_simulation)
choisir_scenario()
display(widgets.VBox([
    scenario,
    widgets.Box(positions, layout=widgets.Layout(display="flex", flex_flow="row wrap")),
    widgets.HBox([tirages, lancer]),
    sortie,
]))
lancer_simulation()

## Exploiter les résultats

1. Retrouve à la main les focales nominales des deux premiers scénarios. Dans quel cas l’histogramme est-il le plus étalé ? Appuie ton explication sur le dénominateur $p-p'$.
2. Pour chaque scénario, compare l’incertitude estimée par simulation et celle obtenue par propagation linéaire. Que changes-tu si tu veux vérifier la stabilité de l’estimation numérique ?
3. Passe au troisième scénario, puis modifie seulement $u(p)$ ou $u(p')$. Quelle mesure gagnerais-tu à rendre plus précise ? Justifie à partir de tes résultats.
4. Modifie la graine dans la fonction de simulation, puis relance les cellules. Quelles valeurs changent ?

*Adapté de « Mesures et Incertitudes », activité 5.*